# Notebook 15 - nnU-Net + Mean Teacher SSL (UNM, 25% labels, MATCHED 150 epochs)

Clean matched comparison: the SAME custom trainer is run twice, the only
difference is the consistency loss (SSL) switched on or off via MT_LAMBDA:

- **Supervised control** = MT_LAMBDA=0  -> pure nnU-Net (no teacher, no unlabeled).
- **Mean Teacher**      = MT_LAMBDA=0.05 -> SSL active.

Both run 150 epochs on the 25% split (66 train / 44 val). Everything else
(LR, schedule, augmentation, split) is identical, so the delta isolates SSL.

Order: setup -> prep unlabeled -> 25% split -> train+eval MT -> train+eval
supervised -> compare. Training cells are long (~6-7h each) and AUTO-RESUME
(re-run the cell if Colab disconnects). Reported as a complementary single-run.


In [ ]:
# === Cell 1: Setup (mount, install nnU-Net v2, env vars) ===
from google.colab import drive
drive.mount('/content/drive')

!pip install -q nnunetv2

import os, glob
BASE      = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
WORKSPACE = os.path.join(BASE, 'nnunet_workspace')
SSL_DIR   = os.path.join(BASE, 'tesis_seg', 'nnunet_ssl')
os.environ['nnUNet_raw']          = os.path.join(WORKSPACE, 'nnUNet_raw')
os.environ['nnUNet_preprocessed'] = os.path.join(WORKSPACE, 'nnUNet_preprocessed')
os.environ['nnUNet_extTrainer']   = SSL_DIR
print('Dataset501 (labeled) preprocessed:',
      bool(glob.glob(os.path.join(os.environ['nnUNet_preprocessed'], 'Dataset501*'))))


In [ ]:
# === Cell 2: Prep Dataset505 (unlabeled r10 pool). Skips itself if already done. ===
import os, json, shutil
import numpy as np
from PIL import Image

BASE      = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
WORKSPACE = os.path.join(BASE, 'nnunet_workspace')
RAW       = os.path.join(WORKSPACE, 'nnUNet_raw')
PP        = os.path.join(WORKSPACE, 'nnUNet_preprocessed')
pp505_2d  = os.path.join(PP, 'Dataset505_VFSS_UNLABELED', 'nnUNetPlans_2d')

if os.path.isdir(pp505_2d) and len([f for f in os.listdir(pp505_2d) if f.endswith(('.npz','.b2nd'))]) > 0:
    print('Dataset505 already preprocessed - skipping.')
else:
    UNLAB_SRC = os.path.join(BASE, 'unlabeling_r10_max0', 'images')
    DS505     = os.path.join(RAW, 'Dataset505_VFSS_UNLABELED')
    imagesTr  = os.path.join(DS505, 'imagesTr'); labelsTr = os.path.join(DS505, 'labelsTr')
    for d in [imagesTr, labelsTr]:
        if os.path.isdir(d): shutil.rmtree(d)
        os.makedirs(d)
    frames = sorted([f for f in os.listdir(UNLAB_SRC) if f.endswith('.png')])
    for fname in frames:
        stem = fname.replace('.png', '')
        img = Image.open(os.path.join(UNLAB_SRC, fname)).convert('L')
        img.save(os.path.join(imagesTr, f'{stem}_0000.png'))
        Image.fromarray(np.zeros((img.height, img.width), dtype=np.uint8)).save(
            os.path.join(labelsTr, f'{stem}.png'))
    json.dump({'channel_names': {'0': 'Xray'}, 'labels': {'background': 0, 'vertebra': 1},
               'numTraining': len(frames), 'file_ending': '.png'},
              open(os.path.join(DS505, 'dataset.json'), 'w'), indent=2)
    pp501 = os.path.join(PP, 'Dataset501_VFSS'); pp505 = os.path.join(PP, 'Dataset505_VFSS_UNLABELED')
    os.makedirs(pp505, exist_ok=True)
    plans = json.load(open(os.path.join(pp501, 'nnUNetPlans.json')))
    plans['dataset_name'] = 'Dataset505_VFSS_UNLABELED'
    json.dump(plans, open(os.path.join(pp505, 'nnUNetPlans.json'), 'w'), indent=2)
    shutil.copy2(os.path.join(DS505, 'dataset.json'), os.path.join(pp505, 'dataset.json'))
    shutil.copy2(os.path.join(pp501, 'dataset_fingerprint.json'),
                 os.path.join(pp505, 'dataset_fingerprint.json'))
    !nnUNetv2_preprocess -d 505 -plans_name nnUNetPlans -c 2d
    print('Dataset505 preprocessed.')


In [ ]:
# === Cell 3: Set Dataset501 to the 25% labeled split (66 train + 44 val) ===
import os, json
BASE  = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
PP501 = os.path.join(BASE, 'nnunet_workspace', 'nnUNet_preprocessed', 'Dataset501_VFSS')
with open(os.path.join(BASE, 'label_fractions', 'frac_25', 'stems.txt')) as f:
    train25 = sorted(l.strip() for l in f if l.strip())
val = sorted(x.replace('.png', '') for x in os.listdir(os.path.join(BASE, 'val', 'images')) if x.endswith('.png'))
assert len(train25) == 66 and len(val) == 44 and not (set(train25) & set(val))
json.dump([{'train': train25, 'val': val}], open(os.path.join(PP501, 'splits_final.json'), 'w'), indent=2)
print(f'Dataset501 split -> 25%: {len(train25)} train, {len(val)} val')


In [ ]:
# === Cell 4: Train MEAN TEACHER (lambda=0.05, 150 epochs) ===
# LONG run (~6-7h). Re-run this cell to AUTO-RESUME if Colab disconnects.
import os, shutil
BASE      = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
WORKSPACE = os.path.join(BASE, 'nnunet_workspace')
SSL_DIR   = os.path.join(BASE, 'tesis_seg', 'nnunet_ssl')
RESULTS   = os.path.join(WORKSPACE, 'nnUNet_results_mt_25pct_150ep')
os.makedirs(RESULTS, exist_ok=True)
os.environ['nnUNet_raw']          = os.path.join(WORKSPACE, 'nnUNet_raw')
os.environ['nnUNet_preprocessed'] = os.path.join(WORKSPACE, 'nnUNet_preprocessed')
os.environ['nnUNet_results']      = RESULTS
os.environ['nnUNet_extTrainer']   = SSL_DIR
# --- matched schedule (both runs identical except MT_LAMBDA) ---
os.environ['MT_NUM_EPOCHS'] = '150'
os.environ['MT_SEMI_START'] = '20'    # consistency starts at epoch 20 (MT only)
os.environ['MT_WARMUP']     = '10'    # ramp over 10 epochs
os.environ['MT_EMA_DECAY']  = '0.99'
os.environ['MT_LAMBDA']     = '0.05'   # mt
# fallback: copy trainer into variants/ in case nnUNet_extTrainer is not honored
try:
    import nnunetv2
    variants = os.path.join(os.path.dirname(nnunetv2.__file__), 'training', 'nnUNetTrainer', 'variants', 'ssl')
    os.makedirs(variants, exist_ok=True); open(os.path.join(variants, '__init__.py'), 'a').close()
    shutil.copy2(os.path.join(SSL_DIR, 'nnUNetTrainerMeanTeacher.py'),
                 os.path.join(variants, 'nnUNetTrainerMeanTeacher.py'))
except Exception as e:
    print('fallback copy skipped:', e)
fold_dir = os.path.join(RESULTS, 'Dataset501_VFSS', 'nnUNetTrainerMeanTeacher__nnUNetPlans__2d', 'fold_0')
resume = '--c' if os.path.exists(os.path.join(fold_dir, 'checkpoint_latest.pth')) else ''
print('mt: MT_LAMBDA=0.05 epochs=150 | results ->', RESULTS, '| resume:', bool(resume))
!nnUNetv2_train 501 2d 0 -tr nnUNetTrainerMeanTeacher {resume}


In [ ]:
# === Cell 5: Predict + evaluate MEAN TEACHER ===
import os, re, json, shutil
import numpy as np
from PIL import Image
BASE      = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
WORKSPACE = os.path.join(BASE, 'nnunet_workspace')
os.environ['nnUNet_results'] = os.path.join(WORKSPACE, 'nnUNet_results_mt_25pct_150ep')
imagesTs = os.path.join(WORKSPACE, 'nnUNet_raw', 'Dataset501_VFSS', 'imagesTs')
pred_dir = os.path.join(BASE, 'nnunet_baseline_results', 'predictions_mt_25pct_150ep')
if os.path.isdir(pred_dir): shutil.rmtree(pred_dir)
os.makedirs(pred_dir)
!nnUNetv2_predict -i {imagesTs} -o {pred_dir} -d 501 -c 2d -f 0 -tr nnUNetTrainerMeanTeacher
GT = os.path.join(BASE, 'test', 'masks')
gt = {f.replace('.png',''): os.path.join(GT,f) for f in os.listdir(GT) if f.endswith('.png')}
def ns(fn): return re.sub(r'_\d{4}$','',fn.replace('.png',''))
def di(p,g):
    inter=np.sum(p*g); sp=np.sum(p); sg=np.sum(g)
    d=(2.0*inter)/(sp+sg) if (sp+sg)>0 else 1.0
    u=sp+sg-inter; return d, (inter/u if u>0 else 1.0)
pr = {ns(f): os.path.join(pred_dir,f) for f in os.listdir(pred_dir) if f.endswith('.png')}
m = sorted(set(pr)&set(gt)); assert len(m)==len(gt), f'{len(m)} vs {len(gt)}'
f1s=[]; ious=[]
for s in m:
    g=np.array(Image.open(gt[s]).convert('L')); p=np.array(Image.open(pr[s]).convert('L'))
    if p.shape!=g.shape: p=np.array(Image.fromarray(p).resize((g.shape[1],g.shape[0]),Image.NEAREST))
    d,i=di((p>0).astype(np.float32),(g>0).astype(np.float32)); f1s.append(d); ious.append(i)
mf1=float(np.mean(f1s)); miou=float(np.mean(ious))
out=os.path.join(BASE,'nnunet_baseline_results','metrics_mt_25pct_150ep.json')
json.dump({'condition':'mt_25pct_150ep','epochs':150,'mean_f1':mf1,'mean_iou':miou,'num_test_images':len(m)}, open(out,'w'), indent=2)
print(f'mt_25pct_150ep: mean_f1 = {mf1:.4f}  mean_iou = {miou:.4f}  -> {out}')


In [ ]:
# === Cell 6: Train SUPERVISED control (lambda=0, 150 epochs) ===
# LONG run (~6-7h). Re-run this cell to AUTO-RESUME if Colab disconnects.
import os, shutil
BASE      = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
WORKSPACE = os.path.join(BASE, 'nnunet_workspace')
SSL_DIR   = os.path.join(BASE, 'tesis_seg', 'nnunet_ssl')
RESULTS   = os.path.join(WORKSPACE, 'nnUNet_results_sup_25pct_150ep')
os.makedirs(RESULTS, exist_ok=True)
os.environ['nnUNet_raw']          = os.path.join(WORKSPACE, 'nnUNet_raw')
os.environ['nnUNet_preprocessed'] = os.path.join(WORKSPACE, 'nnUNet_preprocessed')
os.environ['nnUNet_results']      = RESULTS
os.environ['nnUNet_extTrainer']   = SSL_DIR
# --- matched schedule (both runs identical except MT_LAMBDA) ---
os.environ['MT_NUM_EPOCHS'] = '150'
os.environ['MT_SEMI_START'] = '20'    # consistency starts at epoch 20 (MT only)
os.environ['MT_WARMUP']     = '10'    # ramp over 10 epochs
os.environ['MT_EMA_DECAY']  = '0.99'
os.environ['MT_LAMBDA']     = '0'   # sup
# fallback: copy trainer into variants/ in case nnUNet_extTrainer is not honored
try:
    import nnunetv2
    variants = os.path.join(os.path.dirname(nnunetv2.__file__), 'training', 'nnUNetTrainer', 'variants', 'ssl')
    os.makedirs(variants, exist_ok=True); open(os.path.join(variants, '__init__.py'), 'a').close()
    shutil.copy2(os.path.join(SSL_DIR, 'nnUNetTrainerMeanTeacher.py'),
                 os.path.join(variants, 'nnUNetTrainerMeanTeacher.py'))
except Exception as e:
    print('fallback copy skipped:', e)
fold_dir = os.path.join(RESULTS, 'Dataset501_VFSS', 'nnUNetTrainerMeanTeacher__nnUNetPlans__2d', 'fold_0')
resume = '--c' if os.path.exists(os.path.join(fold_dir, 'checkpoint_latest.pth')) else ''
print('sup: MT_LAMBDA=0 epochs=150 | results ->', RESULTS, '| resume:', bool(resume))
!nnUNetv2_train 501 2d 0 -tr nnUNetTrainerMeanTeacher {resume}


In [ ]:
# === Cell 7: Predict + evaluate SUPERVISED control ===
import os, re, json, shutil
import numpy as np
from PIL import Image
BASE      = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
WORKSPACE = os.path.join(BASE, 'nnunet_workspace')
os.environ['nnUNet_results'] = os.path.join(WORKSPACE, 'nnUNet_results_sup_25pct_150ep')
imagesTs = os.path.join(WORKSPACE, 'nnUNet_raw', 'Dataset501_VFSS', 'imagesTs')
pred_dir = os.path.join(BASE, 'nnunet_baseline_results', 'predictions_sup_25pct_150ep')
if os.path.isdir(pred_dir): shutil.rmtree(pred_dir)
os.makedirs(pred_dir)
!nnUNetv2_predict -i {imagesTs} -o {pred_dir} -d 501 -c 2d -f 0 -tr nnUNetTrainerMeanTeacher
GT = os.path.join(BASE, 'test', 'masks')
gt = {f.replace('.png',''): os.path.join(GT,f) for f in os.listdir(GT) if f.endswith('.png')}
def ns(fn): return re.sub(r'_\d{4}$','',fn.replace('.png',''))
def di(p,g):
    inter=np.sum(p*g); sp=np.sum(p); sg=np.sum(g)
    d=(2.0*inter)/(sp+sg) if (sp+sg)>0 else 1.0
    u=sp+sg-inter; return d, (inter/u if u>0 else 1.0)
pr = {ns(f): os.path.join(pred_dir,f) for f in os.listdir(pred_dir) if f.endswith('.png')}
m = sorted(set(pr)&set(gt)); assert len(m)==len(gt), f'{len(m)} vs {len(gt)}'
f1s=[]; ious=[]
for s in m:
    g=np.array(Image.open(gt[s]).convert('L')); p=np.array(Image.open(pr[s]).convert('L'))
    if p.shape!=g.shape: p=np.array(Image.fromarray(p).resize((g.shape[1],g.shape[0]),Image.NEAREST))
    d,i=di((p>0).astype(np.float32),(g>0).astype(np.float32)); f1s.append(d); ious.append(i)
mf1=float(np.mean(f1s)); miou=float(np.mean(ious))
out=os.path.join(BASE,'nnunet_baseline_results','metrics_sup_25pct_150ep.json')
json.dump({'condition':'sup_25pct_150ep','epochs':150,'mean_f1':mf1,'mean_iou':miou,'num_test_images':len(m)}, open(out,'w'), indent=2)
print(f'sup_25pct_150ep: mean_f1 = {mf1:.4f}  mean_iou = {miou:.4f}  -> {out}')


In [ ]:
# === Cell 8: Compare (matched 150-epoch runs) ===
import os, json
RES = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_baseline_results'
sup = json.load(open(os.path.join(RES,'metrics_sup_25pct_150ep.json')))['mean_f1']
mt  = json.load(open(os.path.join(RES,'metrics_mt_25pct_150ep.json')))['mean_f1']
print('='*56)
print('nnU-Net SSL comparison (UNM, 25% labels, matched 150 epochs)')
print('='*56)
print(f'nnU-Net supervised (lambda=0)      = {sup:.4f}')
print(f'nnU-Net + Mean Teacher (lambda=.05) = {mt:.4f}')
print(f'delta (MT - supervised)            = {mt-sup:+.4f}')
print('='*56)
print('SSL helps if delta > 0. (Context: full 1000-epoch supervised nnU-Net = 0.8962.)')
